In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 3.3 MB/s eta 0:00:00


In [3]:
!python --version
!curl ipecho.net/plain

Python 3.11.13
34.70.241.205

In [4]:
# https://console.groq.com/docs/api-reference#chat-create
import os
import json
import pandas as pd
import datetime
import time
import random
import re
from groq import Groq


### API and configuration

In [5]:
# Retrieve API key via Colab's secrets
from google.colab import userdata
api_key = userdata.get('GROQ_API_KEY')

In [6]:
# Change working directory and load progress

%cd '/content/drive/MyDrive/Colab Notebooks/DS310_NLP/annotation'
file_input = "comment_25.03.16.csv"
output_path = "docs/llama"
key_index, token_usage, text_idx, text_id = 0, 0, 0, 0
try:
    with open('docs/llama.json', 'r') as json_file:
        progress_data = json.load(json_file) # output type: dictionary

    token_usage = progress_data["token_usage"] if "token_usage" in progress_data else 0
    request_usage = progress_data["request_usage"] if "request_usage" in progress_data else 0
    text_idx = progress_data["text_idx"] if "text_idx" in progress_data else 0
    text_id = progress_data["text_id"] if "text_id" in progress_data else None

except Exception as e:
    print("Error:", e)

token_usage = 0 if token_usage is None or token_usage < 0 else token_usage
text_idx = 0 if text_idx is None or text_idx < 0 else text_idx

print(key_index, token_usage, request_usage, text_idx, text_id, sep='\t\t')
start_point = 0 if text_idx == 0 else text_idx + 1
print(key_index, '\s', '\s', start_point, sep='\t\t')

/content/drive/MyDrive/Colab Notebooks/DS310_NLP/annotation
0		2013		2		9150		tik004862
0		\s		\s		9151


In [7]:
# Initialize client with the 1st key

client = Groq(api_key=api_key)


In [8]:
prompts = []
with open('docs/llm_guideline.md', 'r', encoding='utf-8') as f:
    content = f.read()
    sections = content.split('\n## ') # Split the content into sections based on ""
    sections = [sections[0]] + ['## ' + section for section in sections[1:]]
    choice = 0
    prompts = [sections[choice]]

print(choice)
display(prompts[0])

0


'## Prompt for multi-labels annotation\n\n**Objective**: Classify Vietnamese social media comment into one or more of the 27 emotion categories defined by GoEmotions, plus neutral.\n\n---\n\n### **Label Categories**\n- **amusement** (Giải trí) 😂: Cảm giác buồn cười trước nội dung hài hước, thú vị.\n- **excitement** (Hào hứng) 🤩: Cảm giác vui mừng, phấn khích trước sự kiện, thông tin tích cực.\n- **joy** (Niềm vui) 😀: Cảm giác hạnh phúc, vui vẻ, thoải mái trước tình huống tích cực.\n- **love** (Tình yêu) ❤️: Bày tỏ yêu thương, gắn bó với người, vật, hoặc ý tưởng.\n- **desire** (Mong muốn)\xa0😍: Cảm giác mạnh mẽ về mong muốn có được ai đó hoặc điều gì đó.\n- **optimism** (Lạc quan) 🤞: Hy vọng, tin tưởng vào kết quả tốt đẹp cho bản thân hoặc đối tượng khác.\n- **caring** (Quan tâm) 🤗: Thể hiện tình cảm, sự chăm sóc, hoặc hỗ trợ người khác.\n- **pride** (Tự hào) 😌: Hài lòng, kiêu hãnh về bản thân hoặc người khác.\n- **admiration** (Ngưỡng mộ) 👏: Kính trọng, khâm phục ai đó vì phẩm chất hoặ

## Groq

List models
https://console.groq.com/docs/models

## Massive annotating

In [9]:
stop_point = -1 # default: set any value < 0
print(f'start at index: {start_point}, stop at index: {stop_point}')

start at index: 9151, stop at index: -1


In [10]:
# rate limits: https://console.groq.com/settings/limits
# llama3-70b-8192
RPM = 30
TPM = 6_000
TPD = 500_000
tokens = 0
requests = 0

In [11]:
# Check start and stop points
df = pd.read_csv(file_input)
if stop_point < 0:
    stop_point = len(df) - 1

start_row = df.loc[start_point]
print("Start at index:", start_point)
print(f"Start at, id: {start_row['id'], start_row['text']}")
stop_row = df.loc[stop_point]
print("Stop at index:", stop_point)
print(f"Stop at id: {stop_row['id'], stop_row['text']}")

Start at index: 9151
Start at, id: ('tik004863', 'Tội nghiệp quá, mình khổ thấy gd c này mình còn may mắn lắm rồi')
Stop at index: 25959
Stop at id: ('thr000363', 'Bồ đã dốc lòng yêu một người rồi thì việc chia tay sẽ khiến bồ suy sụp, buồn dữ lắm nhưng mà cô gái oii dù có buồn cỡ nào, đau cỡ nào cũng không được năn nỉ hay van xin họ bất cứ điều gì cả. Nếu đủ yêu, đủ kiên nhẫn, đủ quyết tâm thì họ đã không quyết định buông tay rồi bồ ạ. Cứ buồn, cứ khóc, chấp nhận việc mình và họ đã không còn là gì của nhau nữa rồi, xem lại những tấm hình cũ, đến những nơi có nhiều kỉ niệm. Đừng cầu xin tình cảm từ một người đã muốn rời đi bồ nhé. Chúc bồ sớm move on!!!')


In [12]:
# start_point = 11809 # default: set any value < 0
stop_point = -1 # default: set any value < 0

print(f'start at index: {start_point}, stop at index: {stop_point}')

start at index: 9151, stop at index: -1


In [13]:
# Check start and stop points
import pandas as pd

df = pd.read_csv(file_input)
if start_point < 0:
    start_point = 0
if stop_point < 0:
    stop_point = len(df) - 1

start_row = df.loc[start_point]
print("Start at index:", start_point)
print(f"Start at, id: {start_row['id'], start_row['text']}")
stop_row = df.loc[stop_point]
print("Stop at index:", stop_point)
print(f"Stop at id: {stop_row['id'], stop_row['text']}")

Start at index: 9151
Start at, id: ('tik004863', 'Tội nghiệp quá, mình khổ thấy gd c này mình còn may mắn lắm rồi')
Stop at index: 25959
Stop at id: ('thr000363', 'Bồ đã dốc lòng yêu một người rồi thì việc chia tay sẽ khiến bồ suy sụp, buồn dữ lắm nhưng mà cô gái oii dù có buồn cỡ nào, đau cỡ nào cũng không được năn nỉ hay van xin họ bất cứ điều gì cả. Nếu đủ yêu, đủ kiên nhẫn, đủ quyết tâm thì họ đã không quyết định buông tay rồi bồ ạ. Cứ buồn, cứ khóc, chấp nhận việc mình và họ đã không còn là gì của nhau nữa rồi, xem lại những tấm hình cũ, đến những nơi có nhiều kỉ niệm. Đừng cầu xin tình cảm từ một người đã muốn rời đi bồ nhé. Chúc bồ sớm move on!!!')


In [14]:
# Rate limiting parameters & mutable (dictionary, list, set)
REQUEST_INTERVAL = 60 / RPM # seconds per request
key_status = {}
max_retries = 3
# immutable
last_request_time = 0 # initialize  the last request timestamp
retries = 0


# Retry function with exponential backoff
def retry_with_backoff(api_call, backoff_factor=3, max_wait=60):
    global last_request_time, key_index, client, retries # use global variables to impact value
    while retries < max_retries:
        # rate limiting: check if enough time has passed since the last request
        time_since_last_request = time.time() - last_request_time
        if time_since_last_request < REQUEST_INTERVAL:
            sleep_duration = REQUEST_INTERVAL - time_since_last_request
            tolerance = random.randint(1, 3) # add more sleeping time to avoid detection
            sleep_duration += tolerance
            print(f"Rate limiting: Sleeping for {sleep_duration:.2f} seconds...")
            time.sleep(sleep_duration)

        try:
            response = api_call()
            last_request_time = time.time() # Update the last request timestamp
            return response

        except Exception as e:
            print('Error:', e)
            key_status.update({key_index: e})
            wait_time = min(backoff_factor ** retries + random.uniform(0, 1), max_wait)
            print(f"Retrying in {wait_time:.2f} seconds...")
            time.sleep(wait_time)
            retries += 1
    raise Exception("Max retries exceeded")


In [15]:
def clean_text(text):
    text = re.sub(r'\s+', ' ',  text)
    text = text.replace('.\n', '. ').replace('. \n', '. ').replace('\n', '. ').replace('.<br>', '. ').replace('<br>', '. ')
    # Replace specific HTML entities
    text = text.replace('&gt;', '>').replace('&lt;', '<').replace('&amp;', '&').replace('&quot;', '"').replace('&#39;', "'").replace('&nbsp;', ' ')
    return text


def generate_response(contents=None):
    """
    Annotation by LLM
    Args:
        contents: request content
    Returns:
        response: LLM response
    """

    try:
        response = retry_with_backoff(lambda: client.chat.completions.create(
            messages=[{
                'role': 'user',
                'content': contents,
            }],
            model='llama3-70b-8192',
            temperature=0.5,
            max_tokens=random.randint(45, 50),
            top_p=0.9
        ), backoff_factor=3, max_wait=180)
    except Exception as e:
        print(f"Stop due to error:", e)
        return -1

    return response


def save_progress(row:pd.core.series.Series):
    progress_data = {
        "date_time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "key_index": key_index,
        "key": api_key,
        "token_usage": tokens,
        "request_usage": requests,
        "text_idx": row.name,
        "text_id": row['id']
    }

    # Save progress to JSON file
    with open('docs/llama.json', 'w') as json_file:
        json.dump(progress_data, json_file)

    # Append progress to log file
    with open('docs/llama_log.txt', 'a') as log_file:  # 'a' for append mode
        log_entry = f"{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}, key_index: {key_index}, key: {api_key}, token_usage: {tokens}, request_usage: {requests}, text_idx: {row.name}, text_id: {row['id']}\n"
        log_file.write(log_entry)

    print("Progress saved successfully.")
    return


def save_csv(i, ids: list, texts: list, annotations: list, row=None, type=None):
    df = pd.DataFrame({'id': ids, 'text': texts, 'annotation': annotations})
    # Check saved_folder existence
    saved_folder = os.path.join(os.getcwd(), output_path)
    if not os.path.exists(os.path.join(os.getcwd(), saved_folder)):
        os.makedirs(os.path.join(os.getcwd(), saved_folder))

    # Save
    file_name, ext = os.path.splitext(file_input)
    file_name = f"{file_name}_llama-3_{i}.csv"
    if type == 'backup':
        df.to_csv(os.path.join(saved_folder, f"backup_{file_input}"), encoding='utf-8-sig', index=False)
        print(f"Backup to {'backup_' + file_input}")
    else:
        df.to_csv(os.path.join(saved_folder, f"{file_name}"), encoding='utf-8-sig', index=False)
        print(f"Annotation completed and save to {file_name}")

    if row is not None:
        save_progress(row)


In [16]:
"""
Save: write directly output to csv. List, dataframe is for progress backup
"""
ids, texts, annotations = [], [], []

# for i in range(start_point, stop_point + 1):
for i in range(start_point, start_point + 10):
# for i in range(9190, stop_point + 1):
    row = df.loc[i]
    id = row['id']
    tweet = row['text']
    cleaned_text = clean_text(tweet)
    contents = prompts[0] + f"\n\n{cleaned_text}"
    response = generate_response(contents)
    if response == -1:
        break

    tokens += response.usage.total_tokens
    requests += 1

    # Extract response content
    response = response.choices[0].message.content
    print(i, id, response, sep='\t')
    annotation_parts = response.split('\n')
    label = annotation_parts[0].replace('**Label**: ', '').strip()

    ids.append(id)
    texts.append(tweet)
    annotations.append(label)

    if i % 51 == 50: # Backup every 51 idx
        save_csv(i, ids, texts, annotations, row, type='backup')
        sleep_time = random.randint(25, 40)
        print(f"Play around and sleep for {sleep_time} seconds...")
        time.sleep(10)
        response = generate_response()
        tokens += response.usage.total_tokens
        requests += 1
        time.sleep(sleep_time - 10)

    # Early break
    if stop_point is not None and i == stop_point:
        break

    if tokens > TPD * 0.95:
        print('Rate limit - TPD reached:', tokens)
        break

save_csv(i, ids, texts, annotations, row)

9151	tik004863	**Label**: ["sadness", "gratitude", "relief"]
Rate limiting: Sleeping for 5.00 seconds...
9152	tik004864	**Label**: ["anger", "disapproval", "annoyance"]
Rate limiting: Sleeping for 4.00 seconds...
9153	tik004870	**Label**: ["caring", "desire", "joy"]
Rate limiting: Sleeping for 5.00 seconds...
9154	tik004871	**Label**: ["love", "gratitude", "joy"]
Rate limiting: Sleeping for 3.00 seconds...
9155	tik004872	**Label**: ["sadness", "disapproval"]
Rate limiting: Sleeping for 5.00 seconds...
9156	tik004873	**Label**: ["amusement", "joy"]
Rate limiting: Sleeping for 4.00 seconds...
9157	tik004874	**Label**: ["curiosity", "nervousness"]
Rate limiting: Sleeping for 3.00 seconds...
9158	tik004875	**Label**: ["admiration", "gratitude", "caring"]
Rate limiting: Sleeping for 3.00 seconds...
9159	tik004876	**Label**: ["amusement", "surprise"]
Rate limiting: Sleeping for 4.00 seconds...
9160	tik004877	**Label**: ["amusement", "admiration"]
Annotation completed and save to comment_25.0

In [17]:
# In case stop the previous run suddenly, we need to re-map i and row before saving
for i, row in df.iterrows():
    if row['id'] == ids[-1]:
        print(row)
        break

save_csv(i, ids, texts, annotations, row)

Unnamed: 0                9160
id                   tik004877
text          nhìn chị hiền wa
Name: 9160, dtype: object
Annotation completed and save to comment_25.03.16_llama-3_9160.csv
Progress saved successfully.


In [18]:
key_status

{}